In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    FunctionTransformer,
)
import category_encoders as ce
import random
from collections import Counter

In [3]:
df = pd.read_csv('data/training.csv')
df

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,12/2/2009,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,12/2/2009,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,12/2/2009,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,12/2/2009,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [4]:
target = 'IsBadBuy'
date_col = 'PurchDate'
df[date_col] = pd.to_datetime(df[date_col])
unique_dates = np.sort(df[date_col].unique())

n_dates = len(unique_dates)
train_end_date = unique_dates[int(n_dates * 0.33)]
valid_end_date = unique_dates[int(n_dates * 0.66)]

unique_dates = np.sort(df[date_col].unique())

n_dates = len(unique_dates)
train_end_date = unique_dates[int(n_dates * 0.33)]
valid_end_date = unique_dates[int(n_dates * 0.66)]

train_mask = df[date_col] < train_end_date
valid_mask = (df[date_col] >= train_end_date) & (df[date_col] < valid_end_date)
test_mask  = df[date_col] >= valid_end_date

train_df = df.loc[train_mask].copy()
valid_df = df.loc[valid_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

drop_cols = ['RefId', 'PurchDate']
X_train = X_train.drop(columns=drop_cols)
X_valid = X_valid.drop(columns=drop_cols)
X_test  = X_test.drop(columns=drop_cols)


cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])


cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', ce.CountEncoder(
        normalize=True, handle_unknown=0, handle_missing=0
    ))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    remainder='drop'
)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
X_test  = preprocessor.transform(X_test)
print(len(X_train), len(X_valid), len(X_test))

22772 23911 26300


In [5]:
class Node:
    def __init__(self, X, y, depth=0, max_depth=None):
        self.X = X
        self.y = y
        self.depth = depth
        self.max_depth = max_depth

        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None
        
    def gini(self, y):
        if len(y) == 0:
            return 0
        y_int = y.astype(int)
        counts = np.bincount(y_int)
        probs = counts / counts.sum()
        return 1 - np.sum(probs ** 2)
    
    def std(self, y):
        return np.std(y)
    
    def best_split(self, task="classification", max_features=None):
        n_samples, n_features = self.X.shape

        if max_features is None:
            features = range(n_features)
        else:
            features = np.random.choice(
                n_features, max_features, replace=False
            )

        best_feature, best_threshold = None, None
        best_impurity = float("inf")

        for feature in features:
            unique_vals = np.unique(self.X[:, feature])
            if len(unique_vals) == 1:
                continue

            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2
            for threshold in thresholds:
                left = self.X[:, feature] <= threshold
                right = ~left

                if left.sum() == 0 or right.sum() == 0:
                    continue

                y_left, y_right = self.y[left], self.y[right]

                if task == "classification":
                    impurity = (
                        len(y_left) * self.gini(y_left) +
                        len(y_right) * self.gini(y_right)
                    ) / n_samples
                else:
                    impurity = (
                        len(y_left) * self.std(y_left) +
                        len(y_right) * self.std(y_right)
                    ) / n_samples

                if impurity < best_impurity:
                    best_impurity = impurity
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold


    def random_split(self):
        n_samples, n_features = self.X.shape
        for _ in range(10):
            feature = random.randint(0, n_features - 1)
            min_val, max_val = self.X[:, feature].min(), self.X[:, feature].max()
            if min_val == max_val:
                continue
            threshold = random.uniform(min_val, max_val)
            left_mask = self.X[:, feature] <= threshold
            right_mask = ~left_mask
            if left_mask.sum() > 0 and right_mask.sum() > 0:
                return feature, threshold
        return None, None
    
    def build(self, task="classification", extra_random=False, max_features=None):
        if len(set(self.y)) == 1 or self.depth == self.max_depth or len(self.y) < 2:
            self.make_leaf(task)
            return

        if extra_random:
            feature, threshold = self.random_split()
        else:
            feature, threshold = self.best_split(task, max_features)

        if feature is None:
            self.make_leaf(task)
            return

        self.feature_index = feature
        self.threshold = threshold

        left = self.X[:, feature] <= threshold
        right = ~left

        self.left = Node(self.X[left], self.y[left], self.depth + 1, self.max_depth)
        self.right = Node(self.X[right], self.y[right], self.depth + 1, self.max_depth)

        self.left.build(task, extra_random, max_features)
        self.right.build(task, extra_random, max_features)

    def make_leaf(self, task):
        if task == "classification":
            self.value = Counter(self.y).most_common(1)[0][0]
        else:
            self.value = np.mean(self.y)

    def predict_one(self, x):
        if self.left is None and self.right is None:
            return self.value
        if x[self.feature_index] <= self.threshold:
            return self.left.predict_one(x)
        else:
            return self.right.predict_one(x)
        
    
    def predict_proba_one(self, x, n_classes):
        if self.left is None and self.right is None:
            probs = np.zeros(n_classes)

            y = np.atleast_1d(self.y).astype(int)
            counts = Counter(y)

            for cls, cnt in counts.items():
                if cls < n_classes:
                    probs[cls] = cnt / len(y)

            return probs

        if x[self.feature_index] <= self.threshold:
            return self.left.predict_proba_one(x, n_classes)
        else:
            return self.right.predict_proba_one(x, n_classes)



class MyDecisionTreeClassifier:
    def __init__(self, max_depth=5, extra_random=False):
        self.max_depth = max_depth
        self.extra_random = extra_random
        self.root = None
        self.n_classes = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.root = Node(X, y, depth=0, max_depth=self.max_depth)
        self.root.build(task="classification", extra_random=self.extra_random)

    def predict(self, X):
        return np.array([self.root.predict_one(x) for x in X])

    def predict_proba(self, X):
        return np.array([
            self.root.predict_proba_one(x, self.n_classes)
            for x in X
        ])

    
class MyDecisionTreeRegressor:
    def __init__(self, max_depth=5, extra_random=False):
        self.max_depth = max_depth
        self.extra_random = extra_random
        self.root = None

    def fit(self, X, y):
        self.root = Node(X, y, depth=0, max_depth=self.max_depth)
        self.root.build(task="regression", extra_random=self.extra_random)

    def predict(self, X):
        return np.array([self.root.predict_one(x) for x in X])


In [8]:
from sklearn.metrics import roc_auc_score

def gini_score(y_true, y_pred_proba):
    return 2 * roc_auc_score(y_true, y_pred_proba) - 1


In [12]:
clf_skl = DecisionTreeClassifier(max_depth=5)
clf_skl.fit(X_train, y_train)

y_val_proba_skl = clf_skl.predict_proba(X_valid)[:, 1]
gini_val_skl = gini_score(y_valid, y_val_proba_skl)

print(f"Sklearn DecisionTreeClassifier Gini (val): {gini_val_skl:.4f}")

Sklearn DecisionTreeClassifier Gini (val): 0.2831


In [13]:
model = MyDecisionTreeClassifier(max_depth=5)
model.fit(X_train, y_train)
y_val_proba_my = model.predict_proba(X_valid)[:, 1]
gini_val_my = gini_score(y_valid, y_val_proba_my)
gini_val_my

0.2925072824712476

При сравнении нашего кастомного DecisionTreeClassifier с реализацией sklearn на валидационном наборе данных, наш модуль показал более высокий коэффициент Джини (0.29 против 0.28).
Однако, модаль скайлерна намного быстрее. Потому что релизован на c++. Мой метод поиска разделений более точный и медленный.

## 5. Implement the RandomForestClassifier and check its performance.

In [24]:
class MyRandomForestClassifier:
    def __init__(
        self,
        n_estimators=50,
        max_depth=5,
        max_features="sqrt",
        random_state=42
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.n_classes = None

    def _get_max_features(self, n_features):
        if self.max_features == "sqrt":
            return int(np.sqrt(n_features))
        elif self.max_features == "log2":
            return int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_features

    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.n_classes = len(np.unique(y))
        n_samples, n_features = X.shape
        max_features = self._get_max_features(n_features)

        self.trees = []

        for i in range(self.n_estimators):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            X_boot = X[idx]
            y_boot = y[idx]

            tree = MyDecisionTreeClassifier(
                max_depth=self.max_depth
            )
            tree.n_classes = self.n_classes
            tree.root = Node(
                X_boot,
                y_boot,
                depth=0,
                max_depth=self.max_depth
            )
            tree.root.build(
                task="classification",
                max_features=max_features
            )

            self.trees.append(tree)

    def predict_proba(self, X):
        proba = np.zeros((X.shape[0], self.n_classes))

        for tree in self.trees:
            proba += tree.predict_proba(X)

        return proba / self.n_estimators

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


In [25]:
rf = MyRandomForestClassifier(
    n_estimators=30,
    max_depth=5,
    max_features="sqrt",
    random_state=42
)
rf.fit(X_train, y_train)

y_val_proba_rf = rf.predict_proba(X_valid)[:, 1]
gini_rf = gini_score(y_valid, y_val_proba_rf)

print(f"My RandomForest Gini (val): {gini_rf:.4f}")

My RandomForest Gini (val): 0.3243


## 6.

In [26]:
class MyGBDTClassifier:
    def __init__(
        self,
        n_estimators=50,
        learning_rate=0.1,
        max_depth=3,
        max_features=None,
        random_state=42
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.init_value = 0.0

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        np.random.seed(self.random_state)
        p = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.init_value = np.log(p / (1 - p))

        F = np.full(len(y), self.init_value)

        self.trees = []

        for _ in range(self.n_estimators):
            p = self._sigmoid(F)
            gradients = y - p
            tree = MyDecisionTreeRegressor(
                max_depth=self.max_depth
            )
            tree.root = Node(
                X,
                gradients,
                depth=0,
                max_depth=self.max_depth
            )
            tree.root.build(
                task="regression",
                max_features=self.max_features
            )

            update = tree.predict(X)
            F += self.learning_rate * update

            self.trees.append(tree)

    def predict_proba(self, X):
        F = np.full(X.shape[0], self.init_value)

        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)

        p = self._sigmoid(F)
        return np.vstack([1 - p, p]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)


In [ ]:
gbdt = MyGBDTClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3
)

gbdt.fit(X_train, y_train) 

y_val_proba_gbdt = gbdt.predict_proba(X_valid)[:, 1]
gini_gbdt = gini_score(y_valid, y_val_proba_gbdt)

print(f"My GBDT Gini (val): {gini_gbdt:.4f}")
    

My GBDT Gini (val): 0.3344


## 7.

## Общая идея GBDT

Gradient Boosting Decision Trees (GBDT) строит ансамбль деревьев решений последовательно.  
Каждое новое дерево аппроксимирует градиент функции потерь по предсказаниям предыдущих деревьев.  

Для задачи бинарной классификации чаще всего используется логистическая функция потерь  
(binary cross-entropy).

### Ключевые преимущества GBDT

Высокая точность на табличных данных  
Способность моделировать нелинейные зависимости  
Устойчивость к разнородным признакам  

---

## XGBoost

XGBoost (Extreme Gradient Boosting) является одной из первых высокоэффективных реализаций GBDT.

### Основные особенности

Использование второго порядка оптимизации, включающего градиенты и гессианы  
Эффективная регуляризация с применением L1 и L2 штрафов  
Поддержка подвыборки по объектам и признакам  
Высокая стабильность и воспроизводимость результатов  

### DART (Dropouts meet Additive Regression Trees)

DART представляет собой модификацию бустинга, при которой на каждой итерации случайным образом исключается часть ранее обученных деревьев из ансамбля.

Данный подход является аналогом dropout в нейронных сетях и позволяет уменьшить переобучение, повысить устойчивость ансамбля и снизить зависимость модели от отдельных сильных деревьев.

Недостатком DART является более сложная настройка гиперпараметров и увеличение времени обучения.

---

## LightGBM

LightGBM является высокопроизводительной реализацией GBDT, разработанной компанией Microsoft и ориентированной на скорость обучения и масштабируемость.

### Ключевые особенности

Использование стратегии роста деревьев leaf-wise вместо level-wise  
Высокая скорость обучения и низкое потребление памяти  
Эффективная работа с большими наборами данных  
Применение histogram-based алгоритма разбиения признаков  

### Leaf-wise рост

В отличие от классического роста по уровням, LightGBM на каждом шаге выбирает лист, обеспечивающий наибольшее снижение ошибки, и строит более глубокие и асимметричные деревья.

Это позволяет быстрее снижать значение функции потерь и достигать высокой точности.  
При отсутствии ограничений на глубину и число листьев такая стратегия может приводить к переобучению.

---

## CatBoost

CatBoost является библиотекой градиентного бустинга, разработанной компанией Яндекс и оптимизированной для работы с категориальными признаками.

### Работа с категориальными признаками

CatBoost не требует применения One-Hot Encoding.  
Вместо этого используется механизм Ordered Target Statistics, при котором для каждой категории вычисляются статистики по целевой переменной. Вычисления выполняются в случайном порядке объектов, а для каждого объекта используются только предыдущие наблюдения.

Это позволяет избежать утечки целевой переменной, снизить переобучение и использовать категориальные признаки напрямую.

### Ordered Boosting

В CatBoost применяется упорядоченный бустинг, при котором градиенты для каждого объекта вычисляются только по моделям, не обученным на данном объекте. Это повышает устойчивость и обобщающую способность модели.

---

## Сравнение моделей

| Характеристика | XGBoost | LightGBM | CatBoost |
|---------------|--------|----------|----------|
| Стратегия роста | Level-wise | Leaf-wise | Level-wise |
| Категориальные признаки | Требуют кодирования | Ограниченная поддержка | Нативная поддержка |
| Регуляризация | L1, L2, DART | Ограничения на листья | Ordered boosting |
| Скорость обучения | Средняя | Очень высокая | Средняя |
| Устойчивость к утечке данных | Средняя | Средняя | Высокая |

---

## Особые возможности и важные особенности библиотек

В ходе работы были изучены и использованы характерные особенности каждой библиотеки.

### XGBoost

Оптимизация второго порядка с использованием градиентов и гессианов  
Регуляризация на основе L1 и L2 штрафов  
Подвыборка по объектам и признакам  
Поддержка режима DART, снижающего переобучение  

### LightGBM

Стратегия роста деревьев leaf-wise, ускоряющая снижение ошибки  
Histogram-based разбиение признаков, повышающее скорость обучения  
Возможность контроля переобучения с помощью параметров max_depth и num_leaves  

### CatBoost

Нативная обработка категориальных признаков без применения One-Hot Encoding  
Использование Ordered Target Statistics для предотвращения утечки целевой переменной  
Применение упорядоченного бустинга для повышения устойчивости модели  

## Гиперпараметры

### XGBoost

| Гиперпараметр | Значение по умолчанию | Назначение |
|---------------|---------------------|------------|
| n_estimators | 100 | Количество деревьев в ансамбле |
| max_depth | 6 | Максимальная глубина дерева |
| learning_rate (eta) | 0.3 | Насколько сильно каждое дерево корректирует предсказания |
| subsample | 1 | Доля случайных объектов для обучения каждого дерева |
| colsample_bytree | 1 | Доля признаков, используемых при построении дерева |
| gamma | 0 | Минимальное снижение функции потерь для разделения узла |
| reg_alpha | 0 | L1 регуляризация (контроль переобучения) |
| reg_lambda | 1 | L2 регуляризация (контроль переобучения) |
| booster | gbtree | Тип бустинга: дерево, линейная модель или Dart |
| scale_pos_weight | 1 | Вес для положительного класса (для несбалансированных данных) |

---

### LightGBM

| Гиперпараметр | Значение по умолчанию | Назначение |
|---------------|---------------------|------------|
| n_estimators | 100 | Количество деревьев в ансамбле |
| max_depth | -1 | Максимальная глубина дерева (-1 = без ограничения) |
| learning_rate | 0.1 | Насколько сильно каждое дерево корректирует предсказания |
| num_leaves | 31 | Максимальное количество листьев в дереве |
| subsample | 1 | Доля объектов для построения дерева |
| colsample_bytree | 1 | Доля признаков для построения дерева |
| min_child_samples | 20 | Минимальное число объектов в листе (контроль переобучения) |
| reg_alpha | 0 | L1 регуляризация |
| reg_lambda | 0 | L2 регуляризация |
| boosting_type | gbdt | Тип бустинга: gbdt, dart, goss |

---

### CatBoost

| Гиперпараметр | Значение по умолчанию | Назначение |
|---------------|---------------------|------------|
| iterations | 1000 | Количество деревьев |
| learning_rate | 0.03 | Насколько сильно каждое дерево корректирует предсказания |
| depth | 6 | Глубина каждого дерева |
| l2_leaf_reg | 3 | L2 регуляризация листьев |
| random_strength | 1 | Случайная сила для регуляризации |
| bagging_temperature | 1 | Температура для подвыборки объектов при обучении |
| border_count | 254 | Количество границ для преобразования числовых признаков |
| one_hot_max_size | 2 | Максимальное количество категорий для One-Hot кодирования |
| boosting_type | plain | Тип бустинга: plain или ordered |
| verbose | True | Вывод прогресса обучения |


In [9]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)

y_val_proba_lgbm = lgbm.predict_proba(X_valid)[:, 1]
gini_lgbm = gini_score(y_valid, y_val_proba_lgbm)

print(f"LightGBM Gini (val): {gini_lgbm:.4f}")


[LightGBM] [Info] Number of positive: 2574, number of negative: 20198
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3350
[LightGBM] [Info] Number of data points in the train set: 22772, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.113034 -> initscore=-2.060122
[LightGBM] [Info] Start training from score -2.060122
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_val_proba_xgb = xgb.predict_proba(X_valid)[:, 1]
gini_xgb = gini_score(y_valid, y_val_proba_xgb)

print(f"XGBoost Gini (val): {gini_xgb:.4f}")


XGBoost Gini (val): 0.3688


In [12]:
cat = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
    random_seed=42
)

cat.fit(X_train, y_train)

y_val_proba_cat = cat.predict_proba(X_valid)[:, 1]
gini_cat = gini_score(y_valid, y_val_proba_cat)

print(f"CatBoost Gini (val): {gini_cat:.4f}")


CatBoost Gini (val): 0.3585


Все три модели показали близкие результаты. Лучшей по метрике Gini на валидации оказался XGBoost.  
Причины: XGBoost использует DART и точную оптимизацию второго порядка, что иногда помогает немного лучше подстроиться под данные и уменьшить ошибку на валидации.

In [13]:
y_train_proba_lgb = lgbm.predict_proba(X_train)[:, 1]
y_valid_proba_lgb = lgbm.predict_proba(X_valid)[:, 1]
y_test_proba_lgb  = lgbm.predict_proba(X_test)[:, 1]

gini_train_lgb = gini_score(y_train, y_train_proba_lgb)
gini_valid_lgb = gini_score(y_valid, y_valid_proba_lgb)
gini_test_lgb  = gini_score(y_test, y_test_proba_lgb)

print("LightGBM Gini")
print(f"Train: {gini_train_lgb:.4f}")
print(f"Valid: {gini_valid_lgb:.4f}")
print(f"Test : {gini_test_lgb:.4f}\n")


/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Gini
Train: 0.7562
Valid: 0.3620
Test : 0.3637



In [14]:
y_train_proba_xgb = xgb.predict_proba(X_train)[:, 1]
y_valid_proba_xgb = xgb.predict_proba(X_valid)[:, 1]
y_test_proba_xgb  = xgb.predict_proba(X_test)[:, 1]

gini_train_xgb = gini_score(y_train, y_train_proba_xgb)
gini_valid_xgb = gini_score(y_valid, y_valid_proba_xgb)
gini_test_xgb  = gini_score(y_test, y_test_proba_xgb)

print("XGBoost Gini")
print(f"Train: {gini_train_xgb:.4f}")
print(f"Valid: {gini_valid_xgb:.4f}")
print(f"Test : {gini_test_xgb:.4f}\n")

XGBoost Gini
Train: 0.7668
Valid: 0.3688
Test : 0.3581



In [15]:
y_train_proba_cat = cat.predict_proba(X_train)[:, 1]
y_valid_proba_cat = cat.predict_proba(X_valid)[:, 1]
y_test_proba_cat  = cat.predict_proba(X_test)[:, 1]

gini_train_cat = gini_score(y_train, y_train_proba_cat)
gini_valid_cat = gini_score(y_valid, y_valid_proba_cat)
gini_test_cat  = gini_score(y_test, y_test_proba_cat)

print("CatBoost Gini")
print(f"Train: {gini_train_cat:.4f}")
print(f"Valid: {gini_valid_cat:.4f}")
print(f"Test : {gini_test_cat:.4f}")

CatBoost Gini
Train: 0.6043
Valid: 0.3585
Test : 0.3693


все модели переобучены это видно по очень сильной разнице между тренировочными и остальными данными. На тестовой лучше всех оказался catboost